# Cybersecurity Capstone: Suspicious Login Behaviour Detection
**Domain**: Cybersecurity / Identity & Access Management (IAM)
**Task**: Binary Classification (Routine Login `0` vs. Suspicious Login `1`)
**Duration**: 10-Day Milestone Guided Notebook
**Author**: Antigravity Machine Learning Team

---
### Project Objective
Develop an end-to-end, zero-cost ML solution to investigate authentication telemetry, identify patterns associated with suspicious access (brute-force, credential stuffing, impossible travel, and bot attacks), and deploy a production-grade classifier.


## [Day 1 & Day 2] Problem Definition, Environment Setup & Data Ingestion
We import foundational data science libraries and ingest the Kaggle-standard Risk-Based Authentication (RBA) dataset.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
import joblib

# Load dataset
data_path = os.path.join('..', 'data', 'login_activity_dataset.csv')
if not os.path.exists(data_path):
    data_path = 'data/login_activity_dataset.csv'

df = pd.read_csv(data_path)
print(f'Dataset successfully loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


### Data Quality & Missing Value Inspection
Checking for missing values, duplicates, and class imbalance.


In [ ]:
print('Missing Values Check:')
print(df.isnull().sum())
print('
Duplicate Rows Check:', df.duplicated().sum())
print('
Target Class Distribution:')
print(df['is_suspicious'].value_counts(normalize=True).round(4) * 100)


## [Day 3] Exploratory Data Analysis & Cybersecurity Threat Patterns
Investigating key behavioral signals: failed attempts, login hour, session duration, device and location indicators.


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(data=df, x='failed_attempts', hue='is_suspicious', bins=15, multiple='dodge', shrink=0.8)
plt.title('Failed Attempts by Class (Brute Force Anomaly)')

plt.subplot(1, 2, 2)
sns.kdeplot(data=df[df['is_suspicious']==0]['login_hour'], label='Routine', fill=True, alpha=0.3)
sns.kdeplot(data=df[df['is_suspicious']==1]['login_hour'], label='Suspicious', fill=True, alpha=0.3)
plt.title('Login Hour Density (Off-Peak Attack Detection)')
plt.xlabel('Hour of Day (UTC)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
num_cols = [c for c in df.columns if c not in ['login_id', 'timestamp']]
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Cybersecurity Telemetry')
plt.show()


## [Day 4] Preprocessing Pipeline & Leakage-Free Stratified Split
We isolate train and test sets *before* fitting the scaler to guarantee zero data leakage.


In [ ]:
CORE_FEATURES = ['login_hour', 'failed_attempts', 'session_duration', 'device_change_indicator', 'location_change_indicator']
AUX_FEATURES = ['ip_reputation_score', 'is_weekend']
ALL_FEATURES = CORE_FEATURES + AUX_FEATURES

X = df[ALL_FEATURES]
y = df['is_suspicious']

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ('scaler', StandardScaler(), ['login_hour', 'failed_attempts', 'session_duration', 'ip_reputation_score']),
    ('binary', 'passthrough', ['device_change_indicator', 'location_change_indicator', 'is_weekend'])
])

print(f'Training shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Train suspicious ratio: {y_train.mean():.4f}, Test suspicious ratio: {y_test.mean():.4f}')


## [Day 5 & Day 6] Foundational Model Training & 5-Fold Stratified Cross-Validation
Evaluating **Logistic Regression**, **K-Nearest Neighbors (KNN)**, **Decision Tree**, and **Random Forest**.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(C=1.0, class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=10, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=7, class_weight='balanced', random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipelines = {}
results = []

for name, clf in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc').mean()
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1').mean()
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, 'predict_proba') else y_pred
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc_score = roc_auc_score(y_test, y_prob)
    
    pipelines[name] = pipe
    results.append({
        'Model': name,
        'CV ROC-AUC': round(cv_auc, 4),
        'CV F1': round(cv_f1, 4),
        'Test Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall (Sensitivity)': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(auc_score, 4)
    })

pd.DataFrame(results)


## [Day 7 & Day 8] Evaluation, ROC Curves & Cybersecurity Error Analysis
Analyzing False Positives (user friction) versus False Negatives (breach risk).


In [ ]:
plt.figure(figsize=(8, 5))
for name, pipe in pipelines.items():
    probs = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, 'predict_proba') else pipe.predict(X_test)
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate (Friction)')
plt.ylabel('True Positive Rate (Detection Rate)')
plt.title('Receiver Operating Characteristic (ROC) Comparison')
plt.legend()
plt.show()


In [ ]:
# Confusion Matrix for Selected Model (Logistic Regression)
best_pipe = pipelines['Logistic Regression']
y_pred = best_pipe.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Routine', 'Suspicious'], yticklabels=['Routine', 'Suspicious'])
plt.title('Production Model Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Routine', 'Suspicious']))


## [Day 9 & Day 10] Model Serialization & Sample Inference Demonstration
Saving the trained pipeline for the Streamlit web prototype.


In [ ]:
# Serialize pipeline
os.makedirs('../models', exist_ok=True)
joblib.dump(best_pipe, '../models/best_model.pkl')
print('Trained pipeline serialized to models/best_model.pkl')

# Test Single Event Inference Demo
sample_event = pd.DataFrame([{
    'login_hour': 3,                 # 3:00 AM off-peak
    'failed_attempts': 5,            # 5 failed password attempts
    'session_duration': 1.2,         # 1.2 minutes (rapid probe)
    'device_change_indicator': 1,    # New device
    'location_change_indicator': 1,  # Impossible travel
    'ip_reputation_score': 0.85,     # Tor exit node
    'is_weekend': 0
}])

pred = best_pipe.predict(sample_event)[0]
prob = best_pipe.predict_proba(sample_event)[0][1]

print(f'Verdict: {'SUSPICIOUS' if pred == 1 else 'ROUTINE'} (Threat Probability: {prob*100:.2f}%)')
